# Oncogenomics Biomarker Workbench — walkthrough

> ⚠️ **DEMONSTRATION / SYNTHETIC DATA — RESEARCH AND EDUCATION ONLY — NOT FOR CLINICAL USE**
>
> **This notebook uses fully synthetic transcriptomics-style data for educational and research demonstration purposes only. It is not intended for diagnosis, prognosis, treatment selection, clinical decision-making, cancer prediction, biological discovery, or biomarker validation.**

Reading time: about 5–10 minutes. Every step below calls the project's reusable package
(`onco_workbench`, in `src/`). No analysis logic is re-implemented in this notebook.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

import onco_workbench
from onco_workbench.dashboard.data import load_demo_data
from onco_workbench.data.synthetic import generate_synthetic_dataset
from onco_workbench.disclaimers import DATA_LABEL, SHORT_DISCLAIMER
from onco_workbench.ml.classifier_demo import LIMITATIONS, output_statements, run_ml_demo
from onco_workbench.pipeline import run_comparison, run_qc
from onco_workbench.reporting.manifest import display_path
from onco_workbench.viz import static

pd.set_option("display.precision", 3)
print(f"onco_workbench {onco_workbench.__version__} | {DATA_LABEL}")
print(SHORT_DISCLAIMER)

## 1. Project overview

The workbench demonstrates a transparent, reproducible workflow on **simulated**
transcriptomics-style data:

1. validate the data
2. check its quality
3. compare two arbitrary synthetic groups (`Group_A`, `Group_B`)
4. rank genes with a documented heuristic
5. visualize the results

The data were generated with a **known, planted answer**: 40 of 500 synthetic genes were
deliberately shifted between the groups. A correct workflow should recover them. That
makes every step checkable, and it also means nothing here says anything about real
biology.

## 2. Synthetic data: loading and generation

`load_demo_data()` reads the committed synthetic demo files and validates them. If they
are missing, it raises an error telling you to run `obw generate-data`. Below, the same
generator that `obw generate-data` uses recreates the dataset **in memory** from the
configured seed, to show that it is reproducible. Nothing is written to disk.

In [ ]:
data = load_demo_data()
config = data.config
print("Configuration:", display_path(config.source, config.root), "| seed:", config.seed)
print("Expression matrix:", data.expression.shape, "(synthetic samples x synthetic genes)")

regenerated = generate_synthetic_dataset(config.synthetic, config.seed)
same_ids = list(regenerated.expression.index) == list(data.expression.index)
same_values = np.allclose(
    regenerated.expression.to_numpy(), data.expression.to_numpy(), equal_nan=True, atol=1e-9
)
print("In-memory regeneration matches the committed data:", same_ids and same_values)
print("Planted synthetic signal genes:", int(data.ground_truth["is_signal"].sum()))
data.metadata.head()

## 3. Dataset validation summary

Validation reports every problem at once. It runs 23 checks covering identifiers,
numeric values, missingness, group labels and sizes, and whether batch is confounded with
group.

In [ ]:
print(data.validation.summary())

## 4. Compact quality-control summary

In [ ]:
qc = run_qc(data.expression, data.metadata, config)
pd.DataFrame(qc.summary.as_rows(), columns=["Measure", "Value"])

## 5. PCA visualization

Samples are projected onto the first two principal components. For this visual QC only,
missing values are replaced with gene means and genes are scaled. The simulated batch
offset is visible alongside the group difference. Batches are balanced within groups, so
they are not confounded with group.

In [ ]:
static.plot_pca(qc.pca)

## 6. Group comparison

For every gene: the group means, the difference (**Group_B minus Group_A**), Cohen's d,
a Welch t-test, and Benjamini-Hochberg adjusted p-values. The thresholds are display
settings, not conclusions.

In [ ]:
comparison = run_comparison(data.expression, data.metadata, config, data.ground_truth)
results = comparison.results
c = config.comparison
print(
    f"{c.group_b} vs {c.group_a}: {int(results['p_value'].notna().sum())} genes tested; "
    f"{int(results['meets_thresholds'].sum())} meet the display thresholds "
    f"(BH-adjusted p <= {c.fdr_threshold:g} and |d| >= {c.effect_size_threshold:g})."
)

## 7. Ranked simulated candidate genes

These are the ten highest-ranked **synthetic** genes. The ranking score is
|d| × −log10(adjusted p), a sorting heuristic. The last column shows that they are the
**intentionally implanted synthetic signals**. They are not real genes, biomarkers, or
biological findings.

In [ ]:
planted = data.ground_truth.set_index("gene_id")["is_signal"]
top = results.head(10)[["rank", "gene_id", "mean_diff", "cohens_d", "p_adj", "ranking_score", "direction"]]
top.assign(planted_synthetic_signal=top["gene_id"].map(planted))

## 8. Volcano-style plot

Each point is a synthetic gene. Colored points meet both display thresholds, and the
dashed lines mark the thresholds.

In [ ]:
static.plot_volcano(
    results,
    group_a=c.group_a,
    group_b=c.group_b,
    fdr_threshold=c.fdr_threshold,
    effect_size_threshold=c.effect_size_threshold,
)

## 9. Top-ranked genes across samples

The top-ranked synthetic genes (rows) across the samples, ordered by group (columns).
Each gene is z-scored for display.

In [ ]:
top_ids = results.head(config.ranking.top_n)["gene_id"].tolist()
static.plot_top_gene_heatmap(comparison.normalized, data.aligned, top_ids)

### Workflow check against the planted answer

Because the signal was planted, we can count how much of it the workflow recovers. High
recovery is **expected by construction**, because the planted shifts are large and the
simulated noise is low. It shows the code behaves as designed, not that the method would
work on real data.

In [ ]:
pd.DataFrame(comparison.recovery.as_rows(), columns=["Check", "Result"])

## 10. Optional ML demonstration (educational; NOT a clinical prediction model)

A logistic-regression classifier learns to separate the two **synthetic** groups. All
preprocessing (mean imputation and scaling) sits inside a scikit-learn pipeline fitted
on a stratified **training split only**. The held-out test set is only transformed,
never fitted. A **label-permutation sanity check** refits the same pipeline on shuffled
training labels to show chance-level scores. It is a sanity check, not a biological
benchmark.

In [ ]:
ml = run_ml_demo(data.expression, data.metadata, config)
for statement in output_statements(ml.labels.positive):
    print(statement)
m = ml.metrics
pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "F1", "ROC-AUC"],
        "held-out synthetic test set": [m.accuracy, m.precision, m.recall, m.f1, m.roc_auc],
    }
)

In [ ]:
print(ml.cv.reason)
print(ml.comparison_text())
print()
print("Limitations:")
for text in LIMITATIONS:
    print("-", text)

## 11. Reproducibility, limitations, and next steps

**Reproduce everything from a clone** (inside the project's virtual environment):

```bash
python -m pip install -r requirements-dev.txt -e .   # pinned dependencies
obw generate-data     # regenerate the synthetic demo data (byte-identical for the seed)
obw run-analysis      # tables, figures, report -> outputs/
obw ml-demo           # optional ML demonstration -> outputs/ml_demo/
obw dashboard         # local Streamlit dashboard
```

**Limitations**

- All data are simulated. Gene IDs (`SYN_G…`) are not real genes, and `Group_A` /
  `Group_B` are not diagnoses or clinical categories.
- Apparent group differences and ranked genes reflect **intentionally implanted synthetic
  signals**. They are not cancer biomarkers, diagnostic results, clinical predictions,
  biological findings, or treatment insights.
- The Welch t-test on continuous values is a teaching simplification. Real RNA-seq data
  need count-aware, design-aware methods.
- The ML demo's near-perfect scores are expected by construction, on a test set of only
  20 samples.

See the project [README](../README.md) for installation, the documentation index, the
methodology, and the limitations and ethics notes.

---

*DEMONSTRATION / SYNTHETIC DATA. Research and education only, not for clinical use.*